[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AmolSISL/IMDb-Deep-Learning-Research/blob/main/notebooks/02_Amazon_Sampling.ipynb)

# Notebook 02 — Amazon Reviews 2023 (Electronics): Target-Domain Sample

This notebook builds the **target domain** used in RQ2–RQ4 (Hou et al., 2024):

1. Stream the Electronics review file from the original McAuley Lab release (via the official Hugging Face repository) **without loading it into memory**.
2. Apply the IMDb-equivalent labelling rule: 4–5 stars = positive (1), 1–2 stars = negative (0), 3 stars excluded.
3. Draw a **balanced random sample of 20,000 reviews** (reservoir sampling per class, seed 42) and split it into `test` (10,000), `validation` (5,000) and `finetune_pool` (5,000), as planned in Table 3 of the synopsis.
4. Collect the reviews of **30 frequently reviewed products** for the RQ4 retrieval-augmented summaries.
5. Run a short domain-shift EDA comparing Amazon with IMDb.

**Privacy:** the pseudonymous `user_id` field is dropped on read; only review text, rating and product identifiers are kept.

## 0. Setup

In [ ]:
# ---------------------------------------------------------------
# Project setup: mount Google Drive so every notebook shares the same
# data, models and outputs (folder: MyDrive/customer-voice-intelligence)
# ---------------------------------------------------------------
import os, sys, warnings
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/customer-voice-intelligence"
else:  # running locally
    PROJECT_DIR = os.environ.get("CVI_PROJECT_DIR", os.path.abspath("customer-voice-intelligence"))

for sub in ["data/raw", "data/processed", "outputs/figures", "outputs/tables",
            "outputs/results", "models", "code"]:
    os.makedirs(os.path.join(PROJECT_DIR, sub), exist_ok=True)
sys.path.insert(0, os.path.join(PROJECT_DIR, "code"))
print("Project folder:", PROJECT_DIR)

In [ ]:
import cvi_utils as U
U.set_project(PROJECT_DIR)
import io, gzip, json, random, time
from collections import Counter, defaultdict
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import requests
from sklearn.model_selection import train_test_split
U.plot_style(); U.set_seed(42)

## 1. Configuration

The Electronics file contains tens of millions of reviews. Scanning it completely is possible but slow on Colab, so by default the first `MAX_SCAN` reviews are scanned and the balanced sample is drawn **uniformly at random from that window** (reservoir sampling). Set `MAX_SCAN = None` to scan the whole file. The value used is recorded in `amazon_sampling_log.json` and should be reported as a limitation.

If the download is too slow, download the file manually, upload it to `data/raw/amazon/` on Drive, and set `LOCAL_FILE`. If the file size becomes a constraint, a smaller category (for example `Cell_Phones_and_Accessories`) can be substituted by changing `CATEGORY`. The interim report already anticipates this change, and it should be documented if made.

In [ ]:
CATEGORY = "Electronics"
HF_URL = f"https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw/review_categories/{CATEGORY}.jsonl"
UCSD_URL = f"https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/{CATEGORY}.jsonl.gz"
LOCAL_FILE = os.environ.get("CVI_AMAZON_FILE")      # e.g. U.P("data", "raw", "amazon", "Electronics.jsonl.gz")

MAX_SCAN = 5_000_000          # reviews to scan (None = whole file)
PER_CLASS = 10_000            # 10,000 positive + 10,000 negative = 20,000
RESERVOIR = int(PER_CLASS * 1.05)   # small buffer for de-duplication
MIN_TOKENS = 3                # drop empty / one-word texts
N_PRODUCTS = 30               # products for RAG summaries (RQ4)
MIN_REVIEWS_PRODUCT, MIN_NEG_PRODUCT, MAX_PER_PRODUCT = 60, 10, 80

if U.QUICK:   # smoke-test settings
    MAX_SCAN, PER_CLASS, RESERVOIR, N_PRODUCTS, MIN_REVIEWS_PRODUCT, MIN_NEG_PRODUCT, MAX_PER_PRODUCT = 20000, 600, 650, 5, 20, 3, 30

In [ ]:
def open_lines(source):
    """Yield text lines from a local path or URL, transparently handling .gz."""
    if os.path.exists(source):
        f = gzip.open(source, "rt", encoding="utf-8") if source.endswith(".gz") else open(source, encoding="utf-8")
        with f:
            yield from f
        return
    r = requests.get(source, stream=True, timeout=60)
    r.raise_for_status()
    r.raw.decode_content = True
    raw = gzip.GzipFile(fileobj=r.raw) if source.endswith(".gz") else r.raw
    yield from io.TextIOWrapper(raw, encoding="utf-8")


def pick_source():
    if LOCAL_FILE and os.path.exists(LOCAL_FILE):
        return LOCAL_FILE
    for url in (HF_URL, UCSD_URL):
        try:
            r = requests.head(url, allow_redirects=True, timeout=30)
            if r.status_code == 200:
                return url
            print("Not available:", url, r.status_code)
        except Exception as e:
            print("Not reachable:", url, e)
    raise RuntimeError("No Amazon source reachable; download the file manually and set LOCAL_FILE.")

SOURCE = pick_source()
print("Reading from:", SOURCE)

## 2. Pass 1 — balanced reservoir sample
Reservoir sampling (Vitter's Algorithm R) keeps a uniform random sample of fixed size from a stream of unknown length, separately for each class. Product-level review counts are accumulated in the same pass to select the RAG products.

In [ ]:
rng = random.Random(42)
reservoir = {0: [], 1: []}
seen = {0: 0, 1: 0}
rating_counts = Counter()
prod_total, prod_neg = Counter(), Counter()
n_lines = n_bad = n_short = 0
t0 = time.time()

for line in open_lines(SOURCE):
    if MAX_SCAN and n_lines >= MAX_SCAN:
        break
    n_lines += 1
    try:
        r = json.loads(line)
        rating = int(round(float(r["rating"])))
    except Exception:
        n_bad += 1; continue
    rating_counts[rating] += 1
    asin = r.get("parent_asin") or r.get("asin")
    prod_total[asin] += 1
    if rating == 3:
        continue
    label = int(rating >= 4)
    if label == 0:
        prod_neg[asin] += 1
    text = U.clean_text(r.get("text", ""))
    if len(text.split()) < MIN_TOKENS:
        n_short += 1; continue
    rec = (n_lines, asin, rating, label, text, U.clean_text(r.get("title", "")),
           bool(r.get("verified_purchase", False)), int(r.get("helpful_vote", 0) or 0), r.get("timestamp"))
    seen[label] += 1
    k = seen[label]
    if k <= RESERVOIR:
        reservoir[label].append(rec)
    else:
        j = rng.randrange(k)
        if j < RESERVOIR:
            reservoir[label][j] = rec
    if n_lines % 500_000 == 0:
        print(f"  scanned {n_lines:,} reviews in {(time.time() - t0) / 60:.1f} min")

print(f"Scanned {n_lines:,} reviews in {(time.time() - t0) / 60:.1f} min | unreadable {n_bad} | too short {n_short:,}")
print("Eligible: positive", seen[1], "negative", seen[0])
print("Rating counts:", dict(sorted(rating_counts.items())))

## 3. De-duplicate, sample 10,000 per class and split

In [ ]:
cols = ["line_no", "parent_asin", "rating", "label", "review_text", "title", "verified_purchase", "helpful_vote", "timestamp"]
pool = pd.DataFrame(reservoir[0] + reservoir[1], columns=cols)
n_pool = len(pool)
pool = pool.drop_duplicates("review_text")
print(f"Removed {n_pool - len(pool)} exact duplicate texts from the reservoir")

amz = pd.concat([pool[pool.label == l].sample(n=min(PER_CLASS, int((pool.label == l).sum())), random_state=42)
                 for l in (0, 1)], ignore_index=True)
amz["review_id"] = "amz_" + amz.line_no.astype(str)
amz["domain"] = "product"
amz["token_count"] = amz.review_text.str.split().str.len()
amz["has_score"] = amz.review_text.map(U.has_score).astype(int)

n = len(amz)
test, rest = train_test_split(amz, train_size=0.5, stratify=amz.label, random_state=42)
val, pool_ft = train_test_split(rest, train_size=0.5, stratify=rest.label, random_state=42)
amz = pd.concat([test.assign(split="test"), val.assign(split="validation"), pool_ft.assign(split="finetune_pool")], ignore_index=True)

imdb_fence = U.load_imdb().query("split != 'test'").token_count
q1, q3 = imdb_fence.quantile([.25, .75])
amz["is_long_outlier"] = (amz.token_count > q3 + 1.5 * (q3 - q1)).astype(int)

OUT_COLS = ["review_id", "review_text", "rating", "label", "domain", "token_count", "is_long_outlier", "has_score", "split",
            "parent_asin", "title", "verified_purchase", "helpful_vote", "timestamp"]
amz = amz[OUT_COLS]
amz.to_csv(U.P("data", "processed", "amazon_electronics_20k.csv"), index=False)
summary = amz.groupby("split").agg(reviews=("label", "size"), positive_share=("label", "mean"), median_tokens=("token_count", "median"))
U.save_table(summary, "amazon_split_summary", index=True)
print("Saved amazon_electronics_20k.csv:", len(amz), "reviews")
summary

## 4. Pass 2 — reviews of the 30 RAG products (RQ4)
Products are eligible if they have at least `MIN_REVIEWS_PRODUCT` reviews in the scanned window, including at least `MIN_NEG_PRODUCT` negative ones, so that both praise and complaint themes exist. The 30 most-reviewed eligible products are kept, with up to `MAX_PER_PRODUCT` reviews each (all star ratings, including 3 stars).

In [ ]:
eligible = [a for a, c in prod_total.most_common() if c >= MIN_REVIEWS_PRODUCT and prod_neg[a] >= MIN_NEG_PRODUCT]
chosen = eligible[:N_PRODUCTS]
print(f"Eligible products: {len(eligible)}; selected {len(chosen)}")

chosen_set = set(chosen)
store = defaultdict(list)
t0 = time.time(); k = 0
for line in open_lines(SOURCE):
    k += 1
    if MAX_SCAN and k > MAX_SCAN:
        break
    if all(len(store[a]) >= MAX_PER_PRODUCT for a in chosen):
        break
    try:
        r = json.loads(line)
    except Exception:
        continue
    a = r.get("parent_asin") or r.get("asin")
    if a in chosen_set and len(store[a]) < MAX_PER_PRODUCT:
        text = U.clean_text(r.get("text", ""))
        if len(text.split()) >= MIN_TOKENS:
            store[a].append(dict(review_id=f"amzrag_{k}", parent_asin=a, rating=int(round(float(r["rating"]))),
                                 title=U.clean_text(r.get("title", "")), review_text=text))
rag = pd.DataFrame([row for a in chosen for row in store[a]])
rag.to_csv(U.P("data", "processed", "amazon_rag_products.csv"), index=False)
print(f"Pass 2 took {(time.time() - t0) / 60:.1f} min; saved {len(rag):,} reviews for {rag.parent_asin.nunique()} products")
rag.groupby("parent_asin").agg(reviews=("rating", "size"), mean_rating=("rating", "mean")).head()

In [ ]:
log = dict(category=CATEGORY, source=SOURCE, max_scan=MAX_SCAN, scanned=n_lines, unreadable=n_bad, too_short=n_short,
           eligible_pos=seen[1], eligible_neg=seen[0], rating_counts={int(k): v for k, v in rating_counts.items()},
           sample_size=len(amz), rag_products=len(chosen), rag_reviews=len(rag),
           date=time.strftime("%Y-%m-%d"), seed=42)
with open(U.P("outputs", "results", "amazon_sampling_log.json"), "w") as f:
    json.dump(log, f, indent=2)
log

## 5. Domain-shift EDA: Amazon versus IMDb
These figures support RQ2. They show how far the product-review domain differs from the movie-review domain before any model is applied.

In [ ]:
imdb = U.load_imdb()
imdb_tr = imdb[imdb.split == "train"]

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
rc = pd.Series(rating_counts).sort_index()
axes[0].bar(rc.index, rc.values / rc.sum() * 100, color=[U.NEG_COLOR if r <= 2 else ("0.6" if r == 3 else U.POS_COLOR) for r in rc.index])
axes[0].set_xlabel("Star rating"); axes[0].set_ylabel("% of scanned reviews"); axes[0].set_title("(a) Amazon rating distribution\n(before balancing)")

bins = np.linspace(0, 600, 50)
axes[1].hist(imdb_tr.token_count.clip(upper=600), bins=bins, alpha=.55, density=True, label=f"IMDb (median {int(imdb_tr.token_count.median())})", color="#4c566a")
axes[1].hist(amz.token_count.clip(upper=600), bins=bins, alpha=.55, density=True, label=f"Amazon (median {int(amz.token_count.median())})", color="#e09f3e")
axes[1].set_xlabel("Tokens per review (clipped at 600)"); axes[1].set_ylabel("Density"); axes[1].legend(); axes[1].set_title("(b) Review length")

vocab_imdb = U.build_vocab(imdb_tr.review_text, 20000)
amz_tokens = [t for txt in amz[amz.split == "test"].review_text for t in U.tokenize(txt)]
oov_amz = np.mean([t not in vocab_imdb for t in amz_tokens])
imdb_dev_tokens = [t for txt in imdb[imdb.split == "dev_test"].review_text for t in U.tokenize(txt)]
oov_imdb = np.mean([t not in vocab_imdb for t in imdb_dev_tokens])
axes[2].bar(["IMDb dev test", "Amazon test"], [100 * oov_imdb, 100 * oov_amz], color=["#4c566a", "#e09f3e"])
for i, v in enumerate([oov_imdb, oov_amz]):
    axes[2].text(i, 100 * v + 0.1, f"{100 * v:.1f}%", ha="center")
axes[2].set_ylabel("% tokens outside IMDb 20k vocabulary"); axes[2].set_title("(c) Out-of-vocabulary rate")
fig.suptitle("Figure A1. Amazon Electronics versus IMDb: ratings, length and vocabulary", fontweight="bold", y=1.03)
fig.tight_layout(); U.save_fig(fig, "figA1_amazon_domain_shift"); plt.show()

share3 = rating_counts[3] / sum(rating_counts.values())
pos_share = (rating_counts[4] + rating_counts[5]) / (sum(rating_counts.values()) - rating_counts[3])
print(f"3-star share (excluded): {share3:.1%}; positive share among 1/2/4/5 stars: {pos_share:.1%}")
print(f"OOV rate with the IMDb vocabulary: IMDb {oov_imdb:.1%} vs Amazon {oov_amz:.1%}")
print(f"Amazon reviews stating a score like 'x/10': {amz.has_score.mean():.1%} (IMDb {imdb_tr.has_score.mean():.1%})")

In [ ]:
# Words that matter in each domain: top features of a log-odds comparison within Amazon
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
def logodds(df, min_df=10):
    cv = CountVectorizer(tokenizer=U.tokenize, lowercase=False, token_pattern=None, min_df=min_df)
    X = cv.fit_transform(df.review_text); w = np.array(cv.get_feature_names_out())
    yp = np.asarray(X[(df.label == 1).values].sum(0)).ravel() + 0.0; yn = np.asarray(X[(df.label == 0).values].sum(0)).ravel() + 0.0
    a = yp + yn; a0 = a.sum()
    d = np.log((yp + a) / (yp.sum() + a0 - yp - a)) - np.log((yn + a) / (yn.sum() + a0 - yn - a))
    z = d / np.sqrt(1 / (yp + a) + 1 / (yn + a))
    out = pd.DataFrame({"word": w, "z": z})
    return out[~out.word.isin(ENGLISH_STOP_WORDS) & out.word.str.match(r"^[a-z][a-z']+$")]

lo_a = logodds(amz[amz.split != "test"], min_df=10 if not U.QUICK else 3)
lo_i = logodds(imdb_tr, min_df=20 if not U.QUICK else 3)
top_a_neg = lo_a.nsmallest(15, "z"); top_a_pos = lo_a.nlargest(15, "z")
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
axes[0].barh(top_a_neg.word[::-1], top_a_neg.z[::-1], color=U.NEG_COLOR); axes[0].set_title("(a) Negative Amazon reviews")
axes[1].barh(top_a_pos.word[::-1], top_a_pos.z[::-1], color=U.POS_COLOR); axes[1].set_title("(b) Positive Amazon reviews")
for ax in axes: ax.set_xlabel("z-score")
fig.suptitle("Figure A2. Most discriminative words in Amazon Electronics reviews", fontweight="bold", y=1.01)
fig.tight_layout(); U.save_fig(fig, "figA2_amazon_discriminative_words"); plt.show()

k = 200
overlap_neg = len(set(lo_a.nsmallest(k, "z").word) & set(lo_i.nsmallest(k, "z").word)) / k
overlap_pos = len(set(lo_a.nlargest(k, "z").word) & set(lo_i.nlargest(k, "z").word)) / k
print(f"Overlap of the top-{k} sentiment words between domains: negative {overlap_neg:.0%}, positive {overlap_pos:.0%}")

## Summary
* `data/processed/amazon_electronics_20k.csv` contains the balanced target-domain sample, split into `test` (10,000), `validation` (5,000) and `finetune_pool` (5,000).
* `data/processed/amazon_rag_products.csv` contains the product reviews used for the RQ4 summaries.
* Figures A1–A2 quantify the domain shift: product reviews are shorter, use a different vocabulary, and do not state "x/10" scores.

**Next:** `03_RQ1_Architecture_Comparison.ipynb`. Switch the runtime to a **GPU** first.